# 01 · Primeiros Passos com PySpark (Caso A)

**Teoria**: docs/05-pyspark-in-practice.md

Sem Docker — `local[*]` roda Driver e Executors como threads neste mesmo
processo. Este notebook trabalha com um dataset de negócio: `empresas`,
`funcionarios` e `vendas`. O objetivo aqui não é entender COMO o Spark
distribui trabalho (isso vem no notebook 04) — é ganhar fluência nos verbos
que você vai usar o tempo todo: `select`, `filter`, `withColumn`,
`show`/`collect`/`toPandas`.

In [ ]:
import sys

sys.path.insert(0, "../scripts")

from pyspark.sql import SparkSession

# Caso A: modo local puro. Nenhum Docker envolvido — o Spark vem embutido
# no pacote pip. master("local[*]") roda Driver E Executors como threads
# neste mesmo processo — a topologia mais simples que o Spark suporta.
spark = (
    SparkSession.builder.appName("01-primeiros-passos")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark

## Lendo o dataset

Rode `make generate-data SCALE=small` antes, se ainda não gerou os dados.

In [ ]:
from lab_utils import layer_path

empresas = spark.read.parquet(layer_path("local", "bronze", "empresas"))
funcionarios = spark.read.parquet(layer_path("local", "bronze", "funcionarios"))
vendas = spark.read.parquet(layer_path("local", "bronze", "vendas"))

for nome, df in [("empresas", empresas), ("funcionarios", funcionarios), ("vendas", vendas)]:
    print(f"{nome}: {df.count():,} linhas")
    df.printSchema()

## `select` e `withColumn`

`select` escolhe colunas. `withColumn` cria (ou substitui) uma coluna a
partir de uma expressão sobre as demais — aqui, o salário anual a partir do
salário mensal.

In [ ]:
from pyspark.sql.functions import col

funcionarios_resumo = (
    funcionarios
    .select("nome_funcionario", "cargo", "salario")
    .withColumn("salario_anual", col("salario") * 12)
)
funcionarios_resumo.show(10)

## `filter`

Duas perguntas de negócio simples: quais vendas foram grandes (> R\$500)?
Quais funcionários foram contratados nos últimos 2 anos (a partir de
2024-07-01, já que o dataset vai até meados de 2026)?

In [ ]:
vendas_grandes = vendas.filter(col("valor") > 500)
print(f"Vendas acima de R$500: {vendas_grandes.count():,} de {vendas.count():,}")

contratados_recentes = funcionarios.filter(col("data_admissao") >= "2024-07-01")
print(f"Funcionários admitidos desde 2024-07-01: {contratados_recentes.count():,}")
contratados_recentes.select("nome_funcionario", "cargo", "data_admissao").show(10)

## `drop`, `distinct`, `describe`

In [ ]:
setores_unicos = empresas.select("setor").distinct().orderBy("setor")
setores_unicos.show(truncate=False)

cargos_unicos = funcionarios.select("cargo").distinct()
cargos_unicos.show(truncate=False)

vendas.select("valor").describe().show()

funcionarios_sem_id = funcionarios.drop("id_funcionario")
funcionarios_sem_id.show(3)

## `show` vs. `collect` vs. `toPandas`

`show()` imprime uma prévia e não retorna nada usável em Python.
`collect()` traz TODAS as linhas para o Driver como uma lista de `Row`
— cuidado com o tamanho. `toPandas()` faz o mesmo, mas devolve um
DataFrame do Pandas, útil para plotar ou inspecionar em ferramentas que já
conhecem Pandas.

In [ ]:
amostra = empresas.limit(5)

amostra.show()                    # só imprime, retorna None
linhas = amostra.collect()        # lista de pyspark.sql.Row
print(type(linhas), linhas[0])

pdf = amostra.toPandas()          # pandas.DataFrame de verdade
print(type(pdf))
pdf

In [ ]:
spark.stop()